# IceAnalytics — Product Data Science

**Unidad de análisis:** cada fila = un evento `post` (no un usuario). `user_id` se repite.
**Archivos:** `product_activity.csv` → `clean_product_activity.csv` + `quarantine_product_activity.csv` + `metrics_summary.csv`
**Stack:** pandas + matplotlib + seaborn. Código limpio con funciones reutilizables.

> Cómo correr: coloca `product_activity.csv` junto a este notebook y ejecuta de arriba a abajo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

RAW_PATH = "product_activity.csv"
pd.set_option("display.max_columns", None)


## 1. Exploración Inicial
Objetivo: entender volumen, tipos, nulos, duplicados y valores sucios antes de tocar nada.

In [ ]:
def load_raw(path=RAW_PATH):
    df = pd.read_csv(path)
    return df

df_raw = load_raw()
print(f"Shape RAW: {df_raw.shape}")
display(df_raw.head())
print("\n--- INFO ---")
df_raw.info()
print("\n--- DESCRIBE (all) ---")
display(df_raw.describe(include="all"))

In [ ]:
# Nulos y duplicados exactos
nulls = df_raw.isna().sum().sort_values(ascending=False)
print("Conteo de nulos por columna:")
print(nulls[nulls > 0] if (nulls > 0).any() else "Sin nulos detectados")
print(f"\n% nulos por columna (top):\n{(df_raw.isna().mean()*100).sort_values(ascending=False).head(15).round(2)}")

n_dup_exact = int(df_raw.duplicated(keep="first").sum())
print(f"\nDuplicados exactos (filas 100% idénticas): {n_dup_exact}")

In [ ]:
def show_uniques(col, top_n=20):
    print(f"\n===== {col} =====")
    print(f"n_unique (incl. NaN): {df_raw[col].nunique(dropna=False)}")
    print(df_raw[col].value_counts(dropna=False).head(top_n))

for c in ["plan_type", "post_category", "device_type"]:
    if c in df_raw.columns:
        show_uniques(c)
    else:
        print(f"Columna faltante: {c}")

In [ ]:
# Chequeos lógicos de fechas (parseo temporal solo para diagnóstico, sin mutar df_raw)
created_chk = pd.to_datetime(df_raw["created_at"], errors="coerce")
posted_chk = pd.to_datetime(df_raw["post_created_at"], errors="coerce")

n_unparse_created = int(created_chk.isna().sum() - df_raw["created_at"].isna().sum())
n_unparse_posted = int(posted_chk.isna().sum() - df_raw["post_created_at"].isna().sum())
print(f"created_at no parseables (no-nulos que fallan): {n_unparse_created}")
print(f"post_created_at no parseables: {n_unparse_posted}")

mask_valid_dates = created_chk.notna() & posted_chk.notna()
n_pre_signup = int(((posted_chk < created_chk) & mask_valid_dates).sum())
print(f"Posts con post_created_at ANTES de created_at (signup): {n_pre_signup}")

# Inconsistencia de days_since_signup original vs diferencia real
if "days_since_signup" in df_raw.columns:
    real_diff = (posted_chk - created_chk).dt.days
    orig = pd.to_numeric(df_raw["days_since_signup"], errors="coerce")
    comparable = mask_valid_dates & orig.notna() & real_diff.notna()
    mismatches = int((orig[comparable] != real_diff[comparable]).sum())
    denom = int(comparable.sum())
    print(f"days_since_signup comparables: {denom}")
    print(f"Mismatches (orig != real): {mismatches} ({mismatches/denom*100:.2f}% si denom>0)" if denom else "Sin filas comparables")
    # Muestra de mismatches
    display(df_raw.loc[comparable & (orig != real_diff), ["created_at","post_created_at","days_since_signup"]].head())
else:
    print("Columna days_since_signup no existe.")

**Lectura de negocio (exploración):** no tomar decisiones sobre conteos crudos. `plan_type`, `post_category` y `device_type` vienen con mayúsculas, espacios y typos; las fechas mezclan formatos y `days_since_signup` es derivada y poco confiable. Todo lo que sigue limpia antes de medir.

## 2. Limpieza Básica y Quarantine
Reglas: parseo `errors='coerce'` → normalización con diccionarios fijos vía `.map()`/`.replace()` → recálculo de días → `quarantine_df` (errores duros + `reason_code`) vs `core_df` (limpio + deduplicado).

In [ ]:
# 2.1 Conversión a datetime (reporta no parseables)
df = df_raw.copy()
N_RAW = len(df)

df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df["post_created_at"] = pd.to_datetime(df["post_created_at"], errors="coerce")

bad_created = df["created_at"].isna().sum()
bad_posted = df["post_created_at"].isna().sum()
print(f"created_at nulas/no-parseables: {bad_created} ({bad_created/len(df)*100:.2f}%)")
print(f"post_created_at nulas/no-parseables: {bad_posted} ({bad_posted/len(df)*100:.2f}%)")
print("\nEjemplos no parseables (filas crudas):")
mask_bad = df["created_at"].isna() | df["post_created_at"].isna()
display(df_raw.loc[mask_bad, ["created_at","post_created_at","days_since_signup"]].head(10))

In [ ]:
# 2.2 Normalización con diccionarios fijos (.map / .replace)
VALID_PLANS = {"free", "pro", "enterprise"}
VALID_DEVICES = {"web", "mobile", "desktop"}
VALID_CATS = {"tech","life","sports","science","finance","gaming","music","health","education","travel"}

PLAN_FIX = {
    "free":"free","fre":"free","freee":"free","frre":"free","fee":"free","free ":"free",
    "pro":"pro","ppro":"pro","proo":"pro","premium":"pro","prem":"pro","pr0":"pro",
    "enterprise":"enterprise","enterprice":"enterprise","entreprise":"enterprise",
    "ent":"enterprise","entp":"enterprise","corp":"enterprise","business":"enterprise","enterrpise":"enterprise",
}
DEVICE_FIX = {
    "web":"web","weeb":"web","webb":"web","browser":"web",
    "mobile":"mobile","mobil":"mobile","movil":"mobile","phone":"mobile","cell":"mobile",
    "cellphone":"mobile","ios":"mobile","android":"mobile","moblie":"mobile",
    "desktop":"desktop","desk":"desktop","desktopp":"desktop","pc":"desktop",
    "computer":"desktop","mac":"desktop","desktoop":"desktop",
}
CATEGORY_FIX = {
    "tech":"tech","teck":"tech","techn":"tech","technology":"tech","tec":"tech","techh":"tech",
    "life":"life","lif":"life","live":"life","lifestyle":"life","lyfe":"life",
    "sports":"sports","sport":"sports","sportss":"sports","deporte":"sports","spoorts":"sports",
    "science":"science","sciense":"science","scince":"science","sci":"science","sccience":"science",
    "finance":"finance","financ":"finance","fin":"finance","finanzas":"finance","finnance":"finance",
    "gaming":"gaming","gamming":"gaming","game":"gaming","games":"gaming","gameing":"gaming",
    "music":"music","musik":"music","musica":"music","musci":"music","muisc":"music",
    "health":"health","healt":"health","heath":"health","salud":"health","helth":"health",
    "education":"education","edu":"education","educacion":"education","educ":"education","school":"education","educaction":"education",
    "travel":"travel","travell":"travel","traval":"travel","travels":"travel","viaje":"travel","travle":"travel",
}

def normalize_with_map(series, fix_dict):
    """lower+strip y luego .map() al diccionario fijo. Lo no mapeado -> NaN (inválido)."""
    key = series.astype("string").str.lower().str.strip()
    return key.map(fix_dict)

df["plan_clean"] = normalize_with_map(df["plan_type"], PLAN_FIX)
df["device_clean"] = normalize_with_map(df["device_type"], DEVICE_FIX)
df["category_clean"] = normalize_with_map(df["post_category"], CATEGORY_FIX)

print("Distribución limpia vs cruda:")
print("\nplan_clean:"); print(df["plan_clean"].value_counts(dropna=False))
print("\ndevice_clean:"); print(df["device_clean"].value_counts(dropna=False))
print("\ncategory_clean:"); print(df["category_clean"].value_counts(dropna=False))
print("\nValores crudos que quedaron inválidos (top):")
print(df.loc[df['plan_clean'].isna(), 'plan_type'].value_counts(dropna=False).head(10))
print(df.loc[df['device_clean'].isna(), 'device_type'].value_counts(dropna=False).head(10))
print(df.loc[df['category_clean'].isna(), 'post_category'].value_counts(dropna=False).head(10))

In [ ]:
# 2.3 Recálculo obligado + Quarantine con reason_code
df["days_since_signup_calc"] = (df["post_created_at"] - df["created_at"]).dt.days

def build_reason_codes(d):
    reasons = []
    r_created_null = d["created_at"].isna()
    r_posted_null = d["post_created_at"].isna()
    r_pre = d["days_since_signup_calc"] < 0
    r_plan = d["plan_clean"].isna()
    r_dev = d["device_clean"].isna()
    r_cat = d["category_clean"].isna()
    for i in d.index:
        codes = []
        if r_created_null.loc[i]: codes.append("unparseable_created_at")
        if r_posted_null.loc[i]: codes.append("unparseable_post_created_at")
        if not r_created_null.loc[i] and not r_posted_null.loc[i] and r_pre.loc[i]: codes.append("pre_signup_post")
        if r_plan.loc[i]: codes.append("invalid_plan")
        if r_dev.loc[i]: codes.append("invalid_device")
        if r_cat.loc[i]: codes.append("invalid_category")
        reasons.append(";".join(codes) if codes else "ok")
    return reasons

df["reason_code"] = build_reason_codes(df)

quarantine_df = df[df["reason_code"] != "ok"].copy()
core_df = df[df["reason_code"] == "ok"].copy()

# Deduplicado exacto dentro de core (conservar 1 copia) — se reporta aparte
n_dup_in_core = int(core_df.duplicated().sum())
core_df = core_df.drop_duplicates().reset_index(drop=True)
quarantine_df = quarantine_df.reset_index(drop=True)

# Sobrescribir columnas canónicas en core con valores limpios
core_df["plan_type"] = core_df["plan_clean"]
core_df["device_type"] = core_df["device_clean"]
core_df["post_category"] = core_df["category_clean"]

print(f"RAW: {N_RAW} | Quarantine: {len(quarantine_df)} | CORE (dedup): {len(core_df)} | dups removidos en core: {n_dup_in_core}")
display(quarantine_df["reason_code"].value_counts().head(15))
display(core_df.head())

## 3. Data Quality Report
Resumen ejecutivo de calidad: cuánto se descartó, por qué, y cuánto sesgo arrastraba la columna derivada original.

In [ ]:
# Mismatch % de la columna original (sobre filas con ambas fechas válidas y original numérico)
created_ok = pd.to_datetime(df_raw["created_at"], errors="coerce")
posted_ok = pd.to_datetime(df_raw["post_created_at"], errors="coerce")
real_diff = (posted_ok - created_ok).dt.days
orig_num = pd.to_numeric(df_raw["days_since_signup"], errors="coerce") if "days_since_signup" in df_raw.columns else pd.Series(np.nan, index=df_raw.index)
comp_mask = created_ok.notna() & posted_ok.notna() & orig_num.notna()
n_comp = int(comp_mask.sum())
n_mismatch = int((orig_num[comp_mask] != real_diff[comp_mask]).sum()) if n_comp else 0
mismatch_pct = (n_mismatch / n_comp * 100) if n_comp else 0.0

print("===== DATA QUALITY REPORT =====")
print(f"Filas RAW:              {N_RAW}")
print(f"Filas CORE (limpias):   {len(core_df)}")
print(f"Filas Quarantine:       {len(quarantine_df)} ({len(quarantine_df)/N_RAW*100:.2f}% del RAW)")
print(f"Duplicados exactos RAW: {n_dup_exact} ({n_dup_exact/N_RAW*100:.2f}% del RAW)")
print(f"Dups removidos en core: {n_dup_in_core}")
print(f"Inconsistencias days_since_signup: {n_mismatch}/{n_comp} = {mismatch_pct:.2f}%")
print("\nDesglose quarantine por reason_code:")
print((quarantine_df["reason_code"].value_counts(normalize=True)*100).round(2).astype(str) + " %")
print(quarantine_df["reason_code"].value_counts())

**Lectura de negocio:** si Quarantine >5-10% o el mismatch de fechas es alto, ningún KPI de activación/retención previo es confiable. El tablero anterior mezclaba usuarios inválidos, eventos pre-signup y categorías fantasma. A partir de aquí, todo se mide sobre `core_df`.

## 4. Métricas y Análisis (sobre `core_df`)
Volumen = cuántos; Engagement = cuánto valor (votos).

In [ ]:
# 4.1 Volumen: usuarios únicos por plan + actividad (#posts) por país/categoría/dispositivo
print(f"Usuarios únicos (core): {core_df['user_id'].nunique()} | Posts (core): {len(core_df)}")

users_by_plan = core_df.groupby("plan_type")["user_id"].nunique().sort_values(ascending=False)
print("\nUsuarios únicos por plan:"); print(users_by_plan)

def count_plot(series, title):
    order = series.sort_values(ascending=False).index
    plt.figure(figsize=(8,4))
    sns.barplot(x=series.sort_values(ascending=False).values, y=order)
    plt.title(title); plt.xlabel("# posts"); plt.tight_layout(); plt.show()

posts_country = core_df["country"].value_counts()
posts_cat = core_df["post_category"].value_counts()
posts_dev = core_df["device_type"].value_counts()
print("\nPosts por país:"); print(posts_country)
print("\nPosts por categoría:"); print(posts_cat)
print("\nPosts por dispositivo:"); print(posts_dev)
count_plot(posts_country, "Actividad (#posts) por país — core")
count_plot(posts_cat, "Actividad (#posts) por categoría — core")
count_plot(posts_dev, "Actividad (#posts) por dispositivo — core")

In [ ]:
# 4.2 Engagement: votos por plan (media/mediana/percentiles) + por país/categoría/dispositivo
def engagement_table(group_col):
    t = core_df.groupby(group_col)["votes_received"].agg(
        n_posts="count", mean="mean", median="median", std="std",
        p25=lambda s: s.quantile(.25), p75=lambda s: s.quantile(.75),
        p90=lambda s: s.quantile(.90), p95=lambda s: s.quantile(.95),
        total="sum").round(2).sort_values("mean", ascending=False)
    return t

print("Votos por plan:")
display(engagement_table("plan_type"))
print("Votos por país:")
display(engagement_table("country"))
print("Votos por categoría:")
display(engagement_table("post_category"))
print("Votos por dispositivo:")
display(engagement_table("device_type"))

plt.figure(figsize=(8,4))
sns.barplot(data=core_df, x="plan_type", y="votes_received", estimator=np.mean, order=["free","pro","enterprise"])
plt.title("Media de votos por plan (core)"); plt.tight_layout(); plt.show()

### 4.3 Promedios e Interpretación
**Unidad de análisis = evento (fila = post).** El promedio de votos por plan es un promedio *por post*, no *por usuario*: un usuario con 200 posts pesa 200× más que uno con 1 post. **Sesgos/outliers:** usuarios hiperactivos, posts virales (cola derecha), planes con pocos usuarios (alta varianza) y efecto Simpson por país/categoría. Para decisiones de monetización mirar también el promedio a nivel usuario.

In [ ]:
# 4.3 Promedio de votos por plan y posts por usuario
avg_votes_plan = core_df.groupby("plan_type")["votes_received"].mean().round(2)
print("Promedio de votos por post, por plan:"); print(avg_votes_plan)

posts_per_user = core_df.groupby("user_id").size()
print(f"\nPosts por usuario: media={posts_per_user.mean():.2f} | mediana={posts_per_user.median():.2f} | p90={posts_per_user.quantile(.9):.1f} | p99={posts_per_user.quantile(.99):.1f} | max={posts_per_user.max()}")
print(posts_per_user.describe().round(2))

plt.figure(figsize=(7,3.5))
sns.histplot(posts_per_user, bins=50, log_scale=(False, True))
plt.title("Distribución posts por usuario (eje Y log) — cola de hiperactivos"); plt.xlabel("posts/usuario"); plt.tight_layout(); plt.show()

### 4.4 Evento vs Usuario
**Por qué difieren:** el promedio por fila pondera por post (cada post vale 1); el promedio agrupado por usuario pondera por usuario (cada usuario vale 1). Matemáticamente: `mean_fila = sum(votos)/N_posts` vs `mean_usuario = mean(sum(votos)_u / n_posts_u)`. Si los usuarios muy activos tienen votos/post distintos al resto, ambos promedios se separan. El primero responde “¿cómo rinde un post típico?”; el segundo “¿cómo rinde un usuario típico?”. Para producto, reportar ambos.

In [ ]:
# 4.4 Promedio por fila vs promedio agrupado por usuario
mean_per_row = core_df["votes_received"].mean()
per_user_mean = core_df.groupby("user_id")["votes_received"].mean()
mean_of_user_means = per_user_mean.mean()
print(f"Promedio por fila (post): {mean_per_row:.3f}")
print(f"Promedio de promedios por usuario: {mean_of_user_means:.3f}")
print(f"Diferencia: {mean_per_row - mean_of_user_means:+.3f} → {'hiperactivos rinden distinto' if abs(mean_per_row-mean_of_user_means)>1e-9 else 'sin sesgo por actividad'}")
print(per_user_mean.describe([.25,.5,.75,.9,.95,.99]).round(2))

## 5. Concentración, Temporalidad y Bonus

In [ ]:
# Concentración: % posts y % votos del top 1% usuarios más activos
user_stats = core_df.groupby("user_id").agg(n_posts=("post_id","count"), total_votes=("votes_received","sum"))
user_stats = user_stats.sort_values("n_posts", ascending=False)
n_users = len(user_stats)
k = max(1, int(np.ceil(0.01 * n_users)))
top1 = user_stats.head(k)
pct_posts_top1 = top1["n_posts"].sum() / user_stats["n_posts"].sum() * 100
pct_votes_top1 = top1["total_votes"].sum() / user_stats["total_votes"].sum() * 100 if user_stats["total_votes"].sum() else 0
print(f"Usuarios: {n_users} | top1% n={k}")
print(f"Top 1% concentra {pct_posts_top1:.2f}% de posts y {pct_votes_top1:.2f}% de votos")
print(top1.describe().round(2))

plt.figure(figsize=(7,3.5))
sns.histplot(user_stats["n_posts"], bins=50, log_scale=(False, True))
plt.title("Concentración: posts por usuario (Y log)"); plt.tight_layout(); plt.show()

In [ ]:
# Temporalidad: actividad y engagement por mes y por semana
core_df["post_month"] = core_df["post_created_at"].dt.to_period("M").dt.to_timestamp()
core_df["post_week"] = core_df["post_created_at"].dt.to_period("W").dt.start_time

monthly = core_df.groupby("post_month").agg(posts=("post_id","count"), votes=("votes_received","sum"), avg_votes=("votes_received","mean"))
weekly = core_df.groupby("post_week").agg(posts=("post_id","count"), votes=("votes_received","sum"), avg_votes=("votes_received","mean"))
print(monthly.tail(10))

fig, ax = plt.subplots(2,1, figsize=(10,6), sharex=False)
monthly["posts"].plot(ax=ax[0], marker="o"); ax[0].set_title("Posts por mes (core)")
monthly["votes"].plot(ax=ax[1], marker="o", color="orange"); ax[1].set_title("Votos totales por mes (core)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(2,1, figsize=(10,6))
weekly["posts"].plot(ax=ax[0]); ax[0].set_title("Posts por semana (core)")
weekly["avg_votes"].plot(ax=ax[1], color="green"); ax[1].set_title("Voto medio por post, por semana (core)")
plt.tight_layout(); plt.show()

In [ ]:
# Bonus: boxplots de asimetría + cohorte por mes de signup + engagement_score
plt.figure(figsize=(9,4))
sns.boxplot(data=core_df, x="plan_type", y="votes_received", order=["free","pro","enterprise"], showfliers=False)
plt.title("Asimetría de votos por plan (sin outliers para ver la caja)"); plt.tight_layout(); plt.show()

plt.figure(figsize=(10,4))
sns.boxplot(data=core_df, x="post_category", y="votes_received", showfliers=False)
plt.xticks(rotation=30); plt.title("Asimetría de votos por categoría"); plt.tight_layout(); plt.show()
print(f"Skew global votos: {core_df['votes_received'].skew():.2f} (>>0 = cola derecha / virales)")

# engagement_score a nivel evento: premia votos, penaliza post tardío (contenido viejo sin tracción = menos score)
core_df["engagement_score"] = core_df["votes_received"] / np.log1p(1 + core_df["days_since_signup_calc"].clip(lower=0))
display(core_df[["votes_received","days_since_signup_calc","engagement_score"]].describe().round(2))

# Cohorte por mes de signup: actividad y voto medio + matriz de retención de actividad
core_df["signup_month"] = core_df["created_at"].dt.to_period("M").dt.to_timestamp()
cohort = core_df.groupby("signup_month").agg(users=("user_id","nunique"), posts=("post_id","count"), avg_votes=("votes_received","mean"), avg_eng=("engagement_score","mean")).round(2)
print(cohort)

core_df["cohort_index"] = ((core_df["post_month"].dt.year - core_df["signup_month"].dt.year)*12 + (core_df["post_month"].dt.month - core_df["signup_month"].dt.month))
cohort_matrix = core_df.pivot_table(index="signup_month", columns="cohort_index", values="post_id", aggfunc="count", fill_value=0)
plt.figure(figsize=(10,4))
sns.heatmap(cohort_matrix, annot=False, cmap="Blues")
plt.title("Cohortes: posts por mes de signup (filas) vs meses desde signup (cols)"); plt.tight_layout(); plt.show()

## 6. Product Decisions

### ¿Qué segmento priorizarías y por qué?
Priorizar el plan/categoría/país con mayor **voto medio + volumen** en `core_df` (tablas 4.2), no el de más posts brutos. Si `enterprise` tiene el mayor `mean` y `p90` de votos con base de usuarios suficiente, es el segmento de mayor valor por post: duplicar su activación rinde más que inflar posts `free` de bajo voto. A nivel categoría, priorizar la top-2 en `avg_votes × posts` (típicamente `tech/science/finance` en este tipo de productos) y a nivel país, el que combine más posts con voto medio alto. Si el top 1% concentra >30-40% de posts/votos (sección 5), la prioridad es retener a esos creadores, no adquirir usuarios fríos.

### ¿Qué parte del tablero “mentía” antes de la limpieza?
1) **Volumen inflado:** duplicados exactos y eventos pre-signup (`post_created_at < created_at`) contaban como actividad. 2) **`days_since_signup` original:** el % de mismatches (sección 3) invalida cualquier funnel de activación/retención D7/D30 previo. 3) **Segmentación rota:** typos (`teck`, `musik`, `enterprice`, `mobil`) creaban planes/categorías fantasma que fragmentaban el tablero y ocultaban el rendimiento real de `pro/enterprise` y de `tech/music`. 4) **Promedios por post:** sin distinguir evento vs usuario (4.4), los hiperactivos sesgaban la media.

### ¿Qué nuevo dato o evento agregarías al tracking?
1) **Eventos de consumo:** `view`, `vote`, `comment`, `share` con `voter_id` y timestamp — hoy solo tenemos `votes_received` acumulado sin saber quién/cuándo vota. 2) **`session_id` + tiempo de exposición** por post para normalizar votos por alcance. 3) **Fuente de signup y experimento/flag** para atribución. 4) **Validación en ingesta:** constraint `post_created_at >= created_at` y enums cerrados de plan/dispositivo/categoría para no volver a ensuciar.

### 2 acciones concretas + limitaciones
**A1 — Programa de retención del top 1%:** si concentran gran share de votos (sección 5), lanzar destacados/notificaciones y medir `avg_eng` por cohorte; éxito = subir `avg_votes` de la cohorte sin subir posts/usuario (calidad, no spam). **A2 — Foco editorial + onboarding por categoría ganadora:** duplicar prompts/plantillas en la categoría con mayor `mean/p90` de votos y redirigir tráfico `web/mobile` según qué dispositivo rinde mejor en 4.2; éxito = `engagement_score` medio por post +2 dígitos en 4 semanas.
**Limitaciones:** sin datos de impresiones no hay tasa de conversión; `votes_received` tiene causalidad inversa (antigüedad → más votos); países/planes pequeños tienen alta varianza; quarantine elimina sesgo pero puede introducir sesgo de selección si el error no es aleatorio; la cohorte por mes de signup es actividad, no retención real sin evento de retorno.

## 7. Exportación de Entregables

In [ ]:
# Consolidar métricas clave y exportar
metrics_summary = engagement_table("plan_type").reset_index().rename(columns={"plan_type":"segment"})
metrics_summary.insert(0, "grain", "plan")
for g in ["country", "post_category", "device_type"]:
    t = engagement_table(g).reset_index().rename(columns={g:"segment"})
    t.insert(0, "grain", g)
    metrics_summary = pd.concat([metrics_summary, t], ignore_index=True)

core_df.to_csv("clean_product_activity.csv", index=False)
quarantine_df.to_csv("quarantine_product_activity.csv", index=False)
metrics_summary.to_csv("metrics_summary.csv", index=False)
print(f"Exportado: clean={len(core_df)} filas, quarantine={len(quarantine_df)} filas, metrics={len(metrics_summary)} filas")